In [1]:
# conda activate ethograph
import re
import natsort
import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path
import ethograph as eto
from movement.io import load_poses
from movement.kinematics import compute_velocity, compute_speed


In [2]:
path = r"C:\Users\aksel\Downloads\trials.nc"

import ethograph as eto


dt = eto.open(path)
dt.itrial(0)

<xarray.DatasetView> Size: 48B
Dimensions:      (segment: 0, individuals: 1)
Coordinates:
  * individuals  (individuals) <U12 48B 'individual_1'
Dimensions without coordinates: segment
Data variables:
    onset_s      (segment) float64 0B ...
    offset_s     (segment) float64 0B ...
    labels       (segment) int32 0B ...
    individual   (segment) <U1 0B ...
Attributes:
    trial:            1
    fps:              200
    ephys_file:       info.rhd
    kilosort_folder:  C:\Users\aksel\Desktop\Freddy_all\ses-000_date-20250527...

#### Scenario 1: Files alinged to trial number, trial id in file name

In [ ]:
# ─── 1. Find media files ───
# Regex patterns may not always work directly, update for your needs.

video_folder = Path("C:/Users/aksel/Desktop/Freddy_all/20250527_02_Freddy")
pose_folder = Path("C:/Users/aksel/Desktop/Freddy_all/ses-000_date-20250527_02_raw/behav/dlc_test")

video_pattern = re.compile(r"^2025\-05\-27_(?P<trial>\d{3})_Freddy\-cam\-(?P<camera>\d{1})$")
pose_pattern = re.compile(r"^2025\-05\-27_(?P<trial>\d{3})_Freddy_DLC_(?P<camera>3Dcam[^/._-]+)$")

video_files = natsort.natsorted(video_folder.glob("*.mp4"))
pose_files = natsort.natsorted(pose_folder.glob("*.csv"))

video_by_trial = {}
for file_path in video_files:
    match = video_pattern.search(file_path.stem)
    video_by_trial.setdefault(int(match["trial"]), []).append(file_path)

pose_by_trial = {}
for file_path in pose_files:
    match = pose_pattern.search(file_path.stem)
    pose_by_trial.setdefault(int(match["trial"]), []).append(file_path)
  
      
# ─── 2. Create session table ───

# 168 trials - construct programmatically
session_table = pd.DataFrame({
    'trial': list(range(1, 169)),
})

# or via import 


# ─── 3. Load trial datasets ───

ds_list = []
for trial in session_table["trial"]:
    trial_pose_files = pose_by_trial.get(trial)
    if not trial_pose_files:
        continue

    ds = load_poses.from_dlc_file(trial_pose_files[0], fps=30)
    ds["velocity"] = compute_velocity(ds.position)
    ds["speed"] = compute_speed(ds.position)
    ds.attrs["trial"] = trial
    ds_list.append(ds)
    

# ─── 4. Create TrialTree ───

dt = eto.from_datasets(ds_list, session_table=session_table)
# Inspect first trial of TrialTree
dt.itrial(0)


# ─── 5. Store media files in session table ───




video_filenames = []
pose_filenames = []
for trial in session_table["trial"]:
    # Note: Session table stores FILENAMES only, not full paths
    video_filenames.append([filename.name for filename in video_by_trial[trial]])
    pose_filenames.append([filename.name for filename in pose_by_trial[trial]])


# Update or create new dt.session
session = dt.session if dt.session else xr.Dataset()
session["video"] = xr.DataArray(
    # video_filnames shape: (168, 2) <-> (trials, cameras)
    video_filenames,
    dims=["trial", "cameras"],
    coords={"trial": session_table["trial"], "cameras": ['1', '2']}
)
    
session["pose"] = xr.DataArray(
    pose_filenames,
    dims=["trial", "cameras"],
    coords={"trial": session_table["trial"], "cameras": ['1', '2']}
)

dt['session'] = xr.DataTree(session)


# ─── 6. Set stream offsets (temporal alignment) ───

#### Scenario 2: Files defined via cosntant offset, fps may vary

In [ ]:
# ─── 1. Find media files ───
# Regex patterns may not always work directly, update for your needs.

video_folder = Path("C:/Users/aksel/Desktop/Freddy_all/20250527_02_Freddy - datetime")
pose_folder = Path("C:/Users/aksel/Desktop/Freddy_all/ses-000_date-20250527_02_raw/behav/dlc_test")

video_pattern = re.compile(r"^2025\-05\-27_[^/\.]+_Freddy\-cam\-(?P<camera>\d{1})$")
video_files = natsort.natsorted(video_folder.glob("*.mp4"))

unique_cameras = sorted({
    m["camera"]
    for f in video_files
    if (m := video_pattern.search(f.stem))
})

assert len(unique_cameras) == len(video_files), "Allow one video file per camera."


session["video"] = xr.DataArray(
    [video_files],
    dims=["cameras"],
)

# Use per device SR and per modality/device fofset to generate timestamps 




## frame rate (or audio_sr, pose_sr)



# pose ...